# INSTALAR DEPENDENCIAS

In [ ]:
!pip install -q langchain langchain-groq groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.1 MB/s eta 0:00:00


# IMPORTS

In [ ]:
import json
import os
from google.colab import drive
from google.colab import userdata
from langchain_groq import ChatGroq

# =====================================================
# IMPORTS LANGCHAIN
# =====================================================

from langchain_core.messages import (SystemMessage, HumanMessage, AIMessage)

# =====================================================
# GRADIO
# =====================================================

import gradio as gr

#API KEY GROQ

In [ ]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("API Key de Groq cargada correctamente")

API Key de Groq cargada correctamente


CARGAR JSON

In [ ]:
# =====================================================
# GOOGLE DRIVE
# =====================================================

drive.mount('/content/drive', force_remount=True)


# =====================================================
# PATH JSON
# =====================================================

json_path = ('/content/drive/MyDrive/TrainModel/training_report.json')

# =====================================================
# CARGAR JSON
# =====================================================

with open(json_path,'r') as f:
    training_report = json.load(f)

print('JSON cargado correctamente.')

# =====================================================
# TEST
# =====================================================

print(training_report.keys())

Mounted at /content/drive
JSON cargado correctamente.
dict_keys(['project', 'dataset', 'models', 'technical_ranking', 'business_ranking', 'best_technical_model', 'best_business_model', 'overfitting_analysis', 'feature_importance', 'cross_validation', 'neural_network', 'business_rules'])


# CREAR LLM

In [ ]:
modelo_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    streaming=True
)

print("LLM listo.")

LLM listo.


CONTEXTO IA

In [ ]:
contexto = json.dumps(training_report, indent=2)
print("Contexto generado.")

Contexto generado.


#SYSTEM PROMPT PROFESIONAL

In [ ]:
# =====================================================
# SYSTEM PROMPT
# =====================================================

SYSTEM_PROMPT = f"""
Eres un analista senior especializado en:

- Machine Learning
- Detección de fraude financiero
- Ciencia de datos
- Riesgo operacional
- Deep Learning
- XGBoost
- Random Forest
- Métricas de clasificación
- Interpretabilidad de modelos

Tu trabajo es analizar un training report
de modelos antifraude y ayudar al usuario
a interpretar resultados técnicos y operacionales.

====================================================
FORMA DE RESPONDER
====================================================

Tu forma de comunicarte:
- Usas un lenguaje claro y cercano.
- Evitas jerga académica innecesaria.
- Explicas conceptos técnicos de forma simple.
- Relacionas métricas y resultados con escenarios reales de negocio.
- Cuando explicas algo complejo, das ejemplos prácticos.
- Eres paciente y profesional.
- Respondes de forma breve y directa por defecto.
- Solo das respuestas extensas si el usuario las solicita explícitamente.

====================================================
REGLAS IMPORTANTES
====================================================

- SOLO puedes responder usando la información disponible en el training_report.
- NO inventes métricas.
- NO inventes modelos.
- NO inventes resultados.
- Si algo no existe en el reporte, debes indicarlo claramente.
- No respondas temas fuera del proyecto.
- No escribas respuestas de más de 3 párrafos salvo que el usuario solicite profundidad.
- Prioriza claridad antes que cantidad.
- Evita repetir información innecesaria.
- Evita disclaimers innecesarios.

====================================================
TIPOS DE ANALISIS QUE PUEDES HACER
====================================================

- Comparar modelos.
- Explicar métricas técnicas.
- Analizar overfitting.
- Analizar impacto operacional.
- Explicar tradeoffs entre negocio y rendimiento técnico.
- Explicar falsas alarmas (FP) y fraudes perdidos (FN).
- Explicar diferencias entre modelos tradicionales y Deep Learning.
- Explicar por qué un modelo fue mejor técnicamente o operacionalmente.

====================================================
TRAINING REPORT
====================================================

{contexto}
"""

#FUNCIÓN STREAMING

In [ ]:
MODELOS = [
    "llama-3.3-70b-versatile",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "qwen/qwen3-32b",
    "llama-3.1-8b-instant"
]

modelo_actual_idx = 0

modelo_llm = ChatGroq(
    model=MODELOS[modelo_actual_idx],
    temperature=0.2,
    streaming=True
)

print(f"✅ Modelo inicial: {MODELOS[modelo_actual_idx]}")

def cambiar_modelo():

    global modelo_actual_idx
    global modelo_llm

    modelo_actual_idx = (modelo_actual_idx + 1) % len(MODELOS)

    nuevo_modelo = MODELOS[modelo_actual_idx]

    modelo_llm = ChatGroq(
        model=nuevo_modelo,
        temperature=0.2,
        streaming=True
    )

    print(f"🔄 Cambiando a: {nuevo_modelo}")

✅ Modelo inicial: llama-3.3-70b-versatile


In [ ]:
def responder_chat(mensaje, historial):

    global modelo_llm

    mensajes_streaming = [
        SystemMessage(content=SYSTEM_PROMPT)
    ]

    for user_msg, bot_msg in historial:
        mensajes_streaming.append(
            HumanMessage(content=user_msg)
        )
        mensajes_streaming.append(
            AIMessage(content=bot_msg)
        )

    mensajes_streaming.append(
        HumanMessage(content=mensaje)
    )

    respuesta_final = ""

    try:

        for chunk in modelo_llm.stream(
            mensajes_streaming
        ):

            if chunk.content:
                respuesta_final += chunk.content
                yield respuesta_final

    except Exception as e:

        print(f"❌ Error: {e}")

        cambiar_modelo()

        yield (
            "Hubo un problema temporal con el modelo. "
            "Por favor intenta nuevamente."
        )

#UI GRADIO

In [ ]:
# =====================================================
# INTERFAZ GRADIO
# =====================================================

chat = gr.ChatInterface(

    fn=responder_chat,

    title="AI Fraud Detection Analyst",

    description="""
Analiza modelos de Machine Learning entrenados
para detección de fraude financiero.

Puedes preguntar sobre:

• Métricas técnicas
• Overfitting
• Precision / Recall / F1
• Impacto operacional
• Falsas alarmas
• Fraudes perdidos
• Comparación entre modelos
• Riesgo de negocio
• Interpretación de resultados
""",

    theme="soft"
)

chat.launch(debug=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c3e077a067eb087a82.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
